# Этап 3. Анализ MovieLens 100K
Цель — проверить исходные файлы и установить, достаточно ли пользователей с историей до границы периода и с положительным откликом после неё. Выполняйте ячейки сверху вниз. Модели на этом этапе не обучаются.

Понадобятся Python 3.10+, pandas, numpy и matplotlib. При необходимости выполните в отдельной ячейке `%pip install pandas numpy matplotlib` и перезапустите ядро. Перед запуском положите архив [ml-100k.zip](https://files.grouplens.org/datasets/movielens/ml-100k.zip) в папку data рядом с рабочим каталогом ядра (путь указан ниже). Архив сохраняйте для личного исследования согласно [условиям GroupLens](https://files.grouplens.org/datasets/movielens/ml-100k-README.txt).

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import zipfile

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

base = Path.cwd()
archive = base / "data" / "ml-100k.zip"
out = base / "results_stage3"
out.mkdir(exist_ok=True)
url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
print("Архив:", archive, "\nРезультаты:", out)

def save_table(frame, name):
    frame.to_csv(out / name, index=False, encoding="utf-8-sig")

## 1. Загрузка и чтение
Используются исходные оценки u.data, справочник пользователей u.user, фильмы u.item и названия жанров u.genre. Готовые разбиения из архива для этого протокола не нужны.

In [ ]:
with zipfile.ZipFile(archive) as bundle:
    def read_member(name, separator, columns):
        with bundle.open("ml-100k/" + name) as stream:
            return pd.read_csv(stream, sep=separator, header=None, names=columns,
                               encoding="latin-1", keep_default_na=False)

    genres = read_member("u.genre", "|", ["genre", "genre_id"]).sort_values("genre_id")
    genre_cols = genres["genre"].tolist()
    ratings = read_member("u.data", "\t", ["user", "item", "rating", "timestamp"])
    users = read_member("u.user", "|", ["user", "age", "gender", "occupation", "zip"])
    movies = read_member("u.item", "|", ["item", "title", "release", "video_release",
                                        "url", *genre_cols])
ratings["time"] = pd.to_datetime(ratings["timestamp"], unit="s", utc=True)
display(ratings.head())

## 2. Аудит и предварительная обработка
Пропуски в необязательных полях справочника фиксируются. Ошибки в ключах, оценках или жанрах останавливают выполнение, чтобы решение об исправлении было явным. Исходные записи сохраняются.

In [ ]:
checks = {
    "строк оценок": len(ratings), "пользователей в оценках": ratings.user.nunique(),
    "фильмов в оценках": ratings.item.nunique(), "строк справочника пользователей": len(users),
    "строк справочника фильмов": len(movies), "первая оценка UTC": ratings.time.min(),
    "последняя оценка UTC": ratings.time.max()
}
errors = {
    "повторных пар пользователь–фильм": ratings.duplicated(["user", "item"]).sum(),
    "повторных ключей пользователей": users.user.duplicated().sum(),
    "повторных ключей фильмов": movies.item.duplicated().sum(),
    "неизвестных пользователей": (~ratings.user.isin(users.user)).sum(),
    "неизвестных фильмов": (~ratings.item.isin(movies.item)).sum(),
    "недопустимых оценок": (~ratings.rating.isin(range(1, 6))).sum(),
    "пропусков времени": ratings.time.isna().sum(),
    "небинарных жанровых меток": (~movies[genre_cols].isin([0, 1])).sum().sum()
}
missing = []
for name, frame in [("ratings", ratings), ("users", users), ("movies", movies)]:
    for col in frame:
        missing.append((name, col, int((frame[col].isna() | frame[col].eq("")).sum())))
save_table(pd.DataFrame(missing, columns=["файл", "поле", "пропуски"]), "missing.csv")
audit = pd.DataFrame({**checks, **errors}.items(), columns=["показатель", "значение"])
save_table(audit, "audit.csv")
display(audit)
if any(errors.values()):
    raise ValueError("Проверьте audit.csv перед продолжением")
ratings = ratings.sort_values("time", kind="stable").reset_index(drop=True)

## 3. Оценки и история пользователей
Длительность истории измеряется между первой и последней зарегистрированными оценками. Выгружаются также пользователи с нулевой длительностью истории — их оценки имеют одну и ту же временную метку.

In [ ]:
distribution = ratings.rating.value_counts().reindex(range(1, 6), fill_value=0).sort_index()
save_table(distribution.rename_axis("оценка").reset_index(name="число"), "rating_counts.csv")
activity = ratings.groupby("user").agg(events=("item", "size"), first=("time", "min"),
                                      last=("time", "max")).reset_index()
activity["span_days"] = (activity["last"] - activity["first"]).dt.total_seconds() / 86400
save_table(activity, "activity.csv")
display(distribution, activity[["events", "span_days"]].describe())
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(activity.events, bins=30, color="#4a6475", edgecolor="white")
ax.set(xlabel="Число оценок одного пользователя", ylabel="Число пользователей")
fig.tight_layout()
fig.savefig(out / "fig_2_1.png", dpi=300)
plt.show()

## 4. Хронологическое разбиение и проверка доступности кандидатов
Границы берутся по позициям 40, 60 и 80 % в отсортированном потоке. Все записи с пограничной меткой входят в следующий период. В каждом из трёх целевых периодов каталог включает только ранее встречавшиеся фильмы. Подсчёт кандидатов использует уникальность пар, проверенную выше. Оцениваем возможность составить десятку и наличие наблюдаемой положительной метки. Последовательно исключаем из положительных событий записи новых пользователей, новых фильмов и пользователей с недостаточным числом кандидатов.

In [ ]:
cuts = [ratings.time.iloc[len(ratings) * p // 100] for p in (40, 60, 80)]
if len(set([ratings.time.min(), *cuts])) != 4:
    raise ValueError("Совпали границы периодов. Требуется проверить временные метки")
bounds = [ratings.time.min(), *cuts, ratings.time.max() + pd.Timedelta(seconds=1)]
names = ["Начальная история", "Обучение", "Валидация", "Тест"]
periods, funnel = [], []
for k, name in enumerate(names):
    start, stop = bounds[k:k+2]
    target = ratings.loc[(ratings.time >= start) & (ratings.time < stop)]
    periods.append([name, start, stop, len(target), target.user.nunique(),
                    target.item.nunique(), int(target.rating.ge(4).sum())])
    if k == 0:
        continue
    history = ratings.loc[ratings.time < start]
    catalog = history.item.unique()
    candidates = len(catalog) - history.groupby("user").item.nunique()
    active = pd.Index(target.user.unique())
    known = active.intersection(candidates.index)
    enough = known[candidates.loc[known].ge(10).to_numpy()]
    positive = target.loc[target.rating.ge(4)]
    known_positive = positive.loc[positive.user.isin(known)]
    available = known_positive.loc[known_positive.item.isin(catalog)]
    eligible = available.loc[available.user.isin(enough)]
    hits = eligible.groupby("user").size()
    mixed = candidates.loc[hits.index].gt(hits)
    funnel.append([name, len(catalog), len(active), len(known), len(enough), len(hits),
                   int(mixed.sum()), len(positive), len(known_positive), len(available),
                   len(eligible)])
periods = pd.DataFrame(periods, columns=["период", "начало UTC", "конец UTC исключительно",
                                        "оценки", "пользователи", "фильмы", "оценки 4–5"])
funnel = pd.DataFrame(funnel, columns=["период", "каталог", "активные", "с историей",
                                      "с историей и ≥10 кандидатами", "допущены к оценке",
                                      "допущены с обеими метками", "положительные события",
                                      "из них у известных пользователей",
                                      "из них в известном каталоге",
                                      "из них у допущенных пользователей"])
save_table(periods, "periods.csv")
save_table(funnel, "eligibility.csv")
display(periods, funnel)
print("Число всех пользователей с историей в одной временной метке:",
      int(activity.span_days.eq(0).sum()))

## 5. Временная схема и сохранение результатов
На схеме длина каждого отрезка соответствует календарной длительности периода. Верхняя строка — общий поток, ниже показано, какая история доступна для каждого прогноза.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#c2c8ca", "#83a997", "#e4b877", "#8ca9cb"]
for k, name in enumerate(names):
    ax.barh(3, bounds[k+1]-bounds[k], left=bounds[k], color=colors[k], height=0.55)
for k in range(1, 4):
    y = 3-k
    ax.barh(y, bounds[k]-bounds[0], left=bounds[0], color=colors[0], height=0.55)
    ax.barh(y, bounds[k+1]-bounds[k], left=bounds[k], color=colors[k], height=0.55)
ax.set_yticks([3, 2, 1, 0], ["Общий поток", "Обучение", "Валидация", "Тест"])
ax.set_xticks(bounds, [b.strftime("%d.%m.%Y") for b in bounds], rotation=25, ha="right")
ax.set_xlabel("Дата UTC. Серым обозначена доступная история")
fig.tight_layout()
fig.savefig(out / "fig_2_2.png", dpi=300)
plt.show()
metadata = {"source": url, "sha256": hashlib.sha256(archive.read_bytes()).hexdigest(),
            "python": platform.python_version(), "pandas": pd.__version__,
            "numpy": np.__version__, "matplotlib": matplotlib.__version__}
(out / "versions.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Сохранено в:", out)
print("\n".join(sorted(p.name for p in out.iterdir())))